In [1]:
from toollib import *

In [1]:
from toollib.unversal import *
from toollib.data_fetcher import *

In [3]:
import logging
logger = logging.getLogger(__name__)

In [2]:
import json

In [3]:
import pandas as pd
# import pymysql

In [4]:
###query_order_info：
# 获取系统信息
sys_dict = get_sys_info('ath', 'inner') #获取国家'ath'系统信息,'inner'方式连接
country_id = sys_dict['country_id']
country_abbr = sys_dict['country_abbr']
merchant_id = sys_dict['merchant_id']
database = sys_dict['db']   #获取database

In [5]:
sample = pd.read_pickle('/home/zengjunyao/to_zjy_模型训练/df_sample_v11.pkl')

In [6]:
country_abbr = 'th'
user,passwd = 'yutiangang', 'W0vtZiHYHrp7hKun' # 泰国

In [7]:
sys_name = 'ath'
dblink='inner'

In [8]:
# 连接数据库
mysql_rule = get_dbcon(sys_name, user, passwd, dblink=dblink)
#mysql_rule = get_dbcon('ath','yutiangang', 'W0vtZiHYHrp7hKun', dblink='inner')

In [46]:
app_order_ids = list(sample['app_order_id'][:10].values)  #选取样本

In [47]:
 # 处理订单列表 (输入order_list=app_order_ids)
app_order_ids = list(sample['app_order_id'][:10].values)
order_id_list = list(set(app_order_ids))
logger.info(f"共计输入{len(app_order_ids)}条数据，清理重复数据后共{len(order_id_list)}条数据")
order_id_list = list2sqlstr(order_id_list)

2025-05-07 10:29:17,299 - toollib.data_fetcher - INFO - 共计输入10条数据，清理重复数据后共10条数据


In [48]:
req_record_tbl = f"(select * from {database}.t_risk_req_record where biz_type !=4 )" if sys_name == 'af' else f"{database}.t_risk_req_record"

In [49]:
# SQL查询基础订单信息
exc_sql = f"""
           select
                "{country_id}" as country_code,
                "{country_abbr.lower()}" as country_abbr,
                "{merchant_id}" as merchant_id,
                case when a.device_type='ios' then 1
                     when a.device_type='android' then 0 end as device_type,            
                a.app_order_id as order_id,                                             
                a.phone_number as phone, 
                a.id_card_number as nid,
                a.user_id as app_user_id,
                a.app_track_id as track_id,
                a.acq_channel,
                a.apply_time,
                cast(jSON_EXTRACT(work_info, '$.workYears') as SIGNED)  as workYears,
                cast(jSON_EXTRACT(work_info, '$.salaryPayFrequency') as SIGNED)  as salaryPayFrequency,
                cast(jSON_EXTRACT(personal_info, '$.email') as char)  as email,
                b.sms_records_url,
                b.app_list_url,
                b.call_records_url,
                b.calendar_records_url,     
                b.device_info,
                c.tx_id,
                c.req_data,
                cr.term,
                cr.days_per_term
           from (select * from {database}.t_app_order where app_order_id in {order_id_list}) a           
           inner  join {database}.t_app_track b on a.app_track_id = b.id                                 
           inner  join {req_record_tbl} c on a.app_order_id = c.biz_id                                  
           inner join {database}.t_contract cr on cr.app_order_id= a.app_order_id                   
           where c.tx_id is not null
    """


In [50]:
#读取sql数据
data_info = pd.read_sql(exc_sql, con=mysql_rule,dtype={"order_id": str})

In [51]:
data_info

,country_code,country_abbr,merchant_id,device_type,order_id,phone,nid,app_user_id,track_id,acq_channel,...,email,sms_records_url,app_list_url,call_records_url,calendar_records_url,device_info,tx_id,req_data,term,days_per_term
0,66,th,ATH,0,1323706215282728960,0854855085,3100800873461,1258062236822757376,2303008,HLOA,...,"""weerapong5085@gmail.com""",https://faimja.unixobi.com/file/download/32248...,https://faimja.unixobi.com/file/download/e49be...,https://faimja.unixobi.com/file/download/6a0fe...,,"{""SIM_state"":""1"",""abis"":""arm64-v8a,armeabi-v7a...",1323706248283512832,"{""txId"": ""1323706248283512832"", ""applyInfo"": {...",1,7
1,66,th,ATH,0,1323703298987352064,0854855085,3100800873461,1258062236822757376,2303008,HLOA,...,"""weerapong5085@gmail.com""",https://faimja.unixobi.com/file/download/32248...,https://faimja.unixobi.com/file/download/e49be...,https://faimja.unixobi.com/file/download/6a0fe...,,"{""SIM_state"":""1"",""abis"":""arm64-v8a,armeabi-v7a...",1323703393522769921,"{""txId"": ""1323703393522769921"", ""applyInfo"": {...",1,7
2,66,th,ATH,0,1323702219562246144,0945540414,3740300555570,1261698652018532352,2302955,APFL,...,"""newbew2013@gmail.com""",https://mocdksvc.supersafecash.com/file/downlo...,https://mocdksvc.supersafecash.com/file/downlo...,https://mocdksvc.supersafecash.com/file/downlo...,,"{""SIM_state"":""1"",""abis"":""armeabi-v7a,armeabi,""...",1323702345286508544,"{""txId"": ""1323702345286508544"", ""applyInfo"": {...",1,7
3,66,th,ATH,0,1323701931170291712,0854381432,3209600232507,1317640458664894464,2302967,VIJG,...,"""kongsakon2221@gmail.com""",https://bienslcx.ueuti.com/file/download/91a65...,https://bienslcx.ueuti.com/file/download/e823b...,https://bienslcx.ueuti.com/file/download/ad180...,,"{""SIM_state"":""1"",""abis"":""arm64-v8a,armeabi-v7a...",1323701968784809984,"{""txId"": ""1323701968784809984"", ""applyInfo"": {...",1,7
4,66,th,ATH,0,1323713280029974528,0818099792,3740200077714,1302890098922577920,2290440,HLOA,...,"""877@outlook.com""",https://faimja.unixobi.com/file/download/18ec0...,https://faimja.unixobi.com/file/download/79394...,https://faimja.unixobi.com/file/download/d5ef3...,,"{""SIM_state"":""1"",""abis"":""arm64-v8a,armeabi-v7a...",1323713437102465024,"{""txId"": ""1323713437102465024"", ""applyInfo"": {...",1,7
5,66,th,ATH,0,1323713280021585920,0818099792,3740200077714,1302890098922577920,2290440,HLOA,...,"""877@outlook.com""",https://faimja.unixobi.com/file/download/18ec0...,https://faimja.unixobi.com/file/download/79394...,https://faimja.unixobi.com/file/download/d5ef3...,,"{""SIM_state"":""1"",""abis"":""arm64-v8a,armeabi-v7a...",1323713455247024128,"{""txId"": ""1323713455247024128"", ""applyInfo"": {...",1,7
6,66,th,ATH,0,1323669295601246208,0930188843,1310800193097,1318174204879134720,2300467,VIJG,...,"""yodsaton1958@gmail.com""",https://bienslcx.ueuti.com/file/download/c43c7...,https://bienslcx.ueuti.com/file/download/bb939...,https://bienslcx.ueuti.com/file/download/a90af...,,"{""SIM_state"":""1"",""abis"":""arm64-v8a,armeabi-v7a...",1323677179810635776,"{""txId"": ""1323677179810635776"", ""applyInfo"": {...",1,7
7,66,th,ATH,0,1323682197271961600,0616546325,3320501227930,1310220729914449920,2302494,VIJG,...,"""phonekongsin67@gmail.com""",https://bienslcx.ueuti.com/file/download/bf663...,https://bienslcx.ueuti.com/file/download/a1a99...,https://bienslcx.ueuti.com/file/download/4f9e4...,,"{""SIM_state"":""1"",""abis"":""arm64-v8a,armeabi-v7a...",1323690013953056768,"{""txId"": ""1323690013953056768"", ""applyInfo"": {...",1,7
8,66,th,ATH,0,1323722163423240192,0992292953,3101202761112,1320966367341928448,2303190,VIJG,...,"""tae.maneejit@gmail.com""",https://bienslcx.ueuti.com/file/download/79b60...,https://bienslcx.ueuti.com/file/download/28e14...,https://bienslcx.ueuti.com/file/download/3b015...,,"{""SIM_state"":""1"",""abis"":""arm64-v8a,armeabi-v7a...",1323722226748841984,"{""txId"": ""1323722226748841984"", ""applyInfo"": {...",1,7
9,66,th,ATH,0,13237221634274344

In [52]:
assert len(data_info) > 0, '输入的订单号无法匹配到符合业务逻辑的订单，请检查订单号！'
logger.info(f"成功查询到进件相关信息,共计{data_info.shape},对应的用户数量{data_info['app_user_id'].nunique()}")

2025-05-07 10:29:33,072 - toollib.data_fetcher - INFO - 成功查询到进件相关信息,共计(10, 23),对应的用户数量7


In [53]:
#获取要选取的用户id的列表：sample_user_id_list
sample_user_id_list = data_info.app_user_id.unique().tolist()  
logger.info(f"剔除异常订单后，还剩余{len(data_info)}订单，对应{len(sample_user_id_list)}个用户")
sample_user_id_list = list2sqlstr(sample_user_id_list)

2025-05-07 10:29:34,071 - toollib.data_fetcher - INFO - 剔除异常订单后，还剩余10订单，对应7个用户


In [54]:
#查询t_app_user中的用户信息
app_user_sql = f"""
            select user_id as app_user_id,  --用户id
            acq_channel,                    --渠道
            source,                        --来源
            create_time as register_time  --注册时间
        from {database}.t_app_user where user_id in {sample_user_id_list}
"""
app_user_data = pd.read_sql(app_user_sql, con=mysql_rule)

In [55]:
# 查询t_customer中的客户信息：customer_data：
customer_sql = f"""
    select 
        user_id as app_user_id,
        acq_channel,
        create_time as fill_time,
        JSON_EXTRACT(extra_info,'$.lineAccount') as lineAccount 
    from {database}.t_customer 
    where user_id in {sample_user_id_list}
    """
customer_data = pd.read_sql(customer_sql, con=mysql_rule)

In [56]:
#客户表和app用户表中合并app_user_id
#合并merge用户信息
data_info = data_info.merge(app_user_data, on=['app_user_id','acq_channel'])
data_info = data_info[data_info['register_time']<data_info['apply_time']].sort_values(by=['order_id','register_time'],ascending=True)
data_info.drop_duplicates(subset=['order_id'],keep='last',inplace=True)
#合并客户信息
data_info = data_info.merge(customer_data,on=['app_user_id','acq_channel'])
data_info = data_info[data_info['fill_time']<data_info['apply_time']].sort_values(by=['order_id','fill_time'],ascending=True)
data_info.drop_duplicates(subset=['order_id'],keep='last',inplace=True)

In [57]:
# 查询t_installment中的分期信息：installment_data
installment_sql = f"""
    select * 
    from {database}.t_installment 
    where user_id in {sample_user_id_list}
    """
installment_data = pd.read_sql(installment_sql, con=mysql_rule).sort_values(by=['create_time'], ascending=True)
logger.info(f"基于user_id关联的放款单,共计{installment_data.shape}")

2025-05-07 10:29:43,255 - toollib.data_fetcher - INFO - 基于user_id关联的放款单,共计(339, 34)


In [58]:
# 查询t_app_order 中的订单信息：order_data
order_sql = f"""
    select * 
    from {database}.t_app_order 
    where user_id in {sample_user_id_list}
    """
order_data = pd.read_sql(order_sql, con=mysql_rule).sort_values(by=['create_time'], ascending=True)
order_data.fillna(np.nan, inplace=True)
logger.info(f"基于user_id关联的进件订单,共计{order_data.shape}")

2025-05-07 10:29:44,271 - toollib.data_fetcher - INFO - 基于user_id关联的进件订单,共计(454, 34)


In [59]:
# 查询t_contract中的合同信息：contract_data
contract_sql = f"""
    select * 
    from {database}.t_contract 
    where user_id in {sample_user_id_list}
    """
contract_data = pd.read_sql(contract_sql, con=mysql_rule).sort_values(by=['create_time'], ascending=True)
logger.info(f"基于user_id关联的合同单,共计{contract_data.shape}")

2025-05-07 10:29:50,332 - toollib.data_fetcher - INFO - 基于user_id关联的合同单,共计(362, 40)


In [60]:
 # 查询设备信息：device_list_data
device_sql = f"""
    select 
        user_id, 
        device_info, 
        create_time,
        update_time 
    from {database}.t_app_track 
    where user_id in {sample_user_id_list} 
    order by create_time
    """
device_list_data = pd.read_sql(device_sql, con=mysql_rule).sort_values(by=['create_time'], ascending=True)
logger.info(f"基于user_id关联设备数据,共计{device_list_data.shape}")

2025-05-07 10:29:51,983 - toollib.data_fetcher - INFO - 基于user_id关联设备数据,共计(347, 4)


In [72]:
#get_user_info函数：处理用户信息
def get_user_info(df):
    detail = df['req_data']
    request_time = None
    try:
        prod = json.loads(detail)
        user_info = prod.get('applyInfo', {})
        user_info.setdefault('productInfo', {})
        user_info.setdefault('userInfo', {})
        request_time = prod.get('requestTime', None) or df['apply_time'].strftime('%Y-%m-%d %H:%M:%S')

        user_info['productInfo']['createTime'] = str(df['apply_time'])
        user_info['productInfo']['term'] = None if (df['term'] is None) or (np.isnan(df['term'])) else int(df['term'])
        user_info['productInfo']['daysPerTerm'] = None if (df['days_per_term'] is None) or (np.isnan(df['days_per_term'])) else int(df['days_per_term'])
        user_info['userInfo']['userRecord']['registerTime'] = str(df['register_time'])
        user_info['userInfo']['userRecord']['createTime'] = str(df['fill_time'])
        user_info['userInfo']['userRecord']['workYears'] = df['workYears']
        user_info['userInfo']['userRecord']['salaryPayFrequency'] = df['salaryPayFrequency']
        user_info['userInfo']['userRecord']['email'] = eval(df['email'])
        user_info['userInfo']['userRecord']['lineAccount'] = eval(df['lineAccount']) if df['lineAccount'] and df['lineAccount'] != 'null' else ''
        device_info = prod['userHidden']['deviceInfo']
        return user_info, device_info, request_time
    except:
        logger.info(f"order_id:{df['order_id']} 解析失败，请检查数据\n{traceback.format_exc()}")
        return None, None, request_time

data_info[['user_info', 'device_info', 'real_apply_time']] = data_info.apply(get_user_info, axis=1, result_type='expand')

In [77]:
#  根据入参下载数据
fetch_url=True
tqdm_disable=False
if fetch_url:
    fetch_oss_data(data_info, 'app_list_url', sys_name,  tqdm_disable=tqdm_disable, tqdm_desc='app')
    fetch_oss_data(data_info, 'sms_records_url', sys_name,  tqdm_disable=tqdm_disable, tqdm_desc='sms')
    fetch_oss_data(data_info, 'call_records_url', sys_name,  tqdm_disable=tqdm_disable, tqdm_desc='call')
    fetch_oss_data(data_info, 'calendar_records_url', sys_name,  tqdm_disable=tqdm_disable, tqdm_desc='calendar')
else:
    data_info['app_list_url_data'] = None
    data_info['sms_records_url_data'] = None
    data_info['call_records_url_data'] = None
    data_info['calendar_records_url_data'] = None
return data_info, installment_data, order_data, contract_data, device_list_data


calendar: 100%|██████████| 10/10 [00:00<00:00, 27539.75it/s]


SyntaxError: 'return' outside function (2774010567.py, line 14)

In [78]:
data_info

,country_code,country_abbr,merchant_id,device_type,order_id,phone,nid,app_user_id,track_id,acq_channel,...,source,register_time,fill_time,lineAccount,user_info,real_apply_time,app_list_url_data,sms_records_url_data,call_records_url_data,calendar_records_url_data
0,66,th,ATH,0,1323669295601246208,0930188843,1310800193097,1318174204879134720,2300467,VIJG,...,1,2024-12-16 18:12:22,2024-12-16 18:13:41,"""boat19394""","{'userInfo': {'ocrRecord': {'code': 'SUCCESS',...",2024-12-31 22:40:32,"[{""app_name"":""โคลนโทรศัพท์"",""app_package"":""com...","[{""body"":""[เงินทันใจ] 4983 คือรหัสยืนยันของคุณ...","[{""call_log_address_name"":""พี่ปุ๊ก TTB"",""call_...",None
1,66,th,ATH,0,1323682197271961600,0616546325,3320501227930,1310220729914449920,2302494,VIJG,...,5,2024-11-29 16:18:29,2024-11-29 16:20:18,"""0616546325""","{'userInfo': {'ocrRecord': {'code': 'SUCCESS',...",2024-12-31 23:31:32,"[{""app_name"":""โคลนโทรศัพท์"",""app_package"":""com...","[{""body"":""รหัสยืนยันของคุณคือ: 3813 รหัสยืนยัน...","[{""call_log_connection_status"":""1"",""call_log_d...",None
2,66,th,ATH,0,1323701931170291712,0854381432,3209600232507,1317640458664894464,2302967,VIJG,...,2,2024-12-15 06:50:19,2024-12-15 06:52:46,"""0854381432""","{'userInfo': {'ocrRecord': {'code': 'SUCCESS',...",2025-01-01 00:19:02,"[{""app_name"":""Tethering"",""app_package"":""com.go...","[{""body"":""GCP เติมเงิน 1000฿"",""phone"":""SMSOTP""...","[{""call_log_address_name"":""ก้อง"",""call_log_con...",None
3,66,th,ATH,0,1323702219562246144,0945540414,3740300555570,1261698652018532352,2302955,APFL,...,2,2024-07-13 21:57:01,2024-07-13 22:00:00,"""new-bew""","{'userInfo': {'ocrRecord': {'code': 'SUCCESS',...",2025-01-01 00:20:32,"[{""app_name"":""Tethering"",""app_package"":""com.go...","[{""body"":""hi89a.com/a \n✅สมัครฟรี8888แจกโบนัส2...","[{""call_log_address_name"":""น้าดวง"",""call_log_c...",None
4,66,th,ATH,0,1323703298987352064,0854855085,3100800873461,1258062236822757376,2303008,HLOA,...,5,2024-09-04 15:27:27,2024-09-04 15:54:30,"""""","{'userInfo': {'ocrRecord': {'code': 'SUCCESS',...",2025-01-01 00:24:42,"[{""app_name"":""Tethering"",""app_package"":""com.go...","[{""body"":""hi89a.com/a \n✅สมัครฟรี8888แจกโบนัส2...","[{""call_log_address_name"":"""",""call_log_connect...",None
5,66,th,ATH,0,1323706215282728960,0854855085,3100800873461,1258062236822757376,2303008,HLOA,...,5,2024-09-04 15:27:27,2024-09-04 15:54:30,"""""","{'userInfo': {'ocrRecord': {'code': 'SUCCESS',...",2025-01-01 00:36:02,"[{""app_name"":""Tethering"",""app_package"":""com.go...","[{""body"":""hi89a.com/a \n✅สมัครฟรี8888แจกโบนัส2...","[{""call_log_address_name"":"""",""call_log_connect...",None
6,66,th,ATH,0,1323713280021585920,0818099792,3740200077714,1302890098922577920,2290440,HLOA,...,2,2024-11-04 13:58:44,2024-11-04 14:00:06,"""""","{'userInfo': {'ocrRecord': {'code': 'SUCCESS',...",2025-01-01 01:04:41,"[{""app_name"":""Clone Phone"",""app_package"":""com....","[{""body"":""พบกับแพ็กเสริมราคาพิเศษ Flash Sale ท...","[{""call_log_address_name"":"""",""call_log_connect...",None
7,66,th,ATH,0,1323713280029974528,0818099792,3740200077714,1302890098922577920,2290440,HLOA,...,2,2024-11-04 13:58:44,2024-11-04 14:00:06,"""""","{'userInfo': {'ocrRecord': {'code': 'SUCCESS',...",2025-01-01 01:04:36,"[{""app_name"":""Clone Phone"",""app_package"":""com....","[{""body"":""พบกับแพ็กเสริมราคาพิเศษ Flash Sale ท...","[{""call_log_address_name"":"""",""call_log_connect...",None
8,66,th,ATH,0,1323722163423240192,0992292953,3101202761112,1320966367341928448,2303190,VIJG,...,2,2024-12-24 11:07:41,2024-12-24 11:08:44,"""Taetae2523""","{'userInfo': {'ocrRecord': {'code': 'SUCCESS',...",2025-01-01 01:39:32,"[{""app_name"":""Tethering"",""app_package"":""com.go...","[{""body"":""[เงินทันใจ] 4339 คือรหัสยืนยันของคุณ...","[{""call_log_address_name"":""โก้ 2824"",""call_log...",None
9,66,th,ATH,0,1323722163427434496,0992292953,3101202761112,1320966367341928448,2303190,VIJG,...,2,2024-12-24 11:07:41,2024-12-24 11:08:44,"""Taetae25

In [73]:
#fdc_data
fdc_data = None

In [80]:
def preprocess_order_data_a(app_order_id, order_data_a):
    """
    预处理order_data_a数据，仅回溯时使用。
    :param app_order_id:申请订单号
    :param order_data_a:申请订单号
    :return:
    """

    app_order_id = int(app_order_id)
    if len(order_data_a) == 0:
        return order_data_a
    else:
        order_data_a_new = [item for item in order_data_a if item['app_order_id'] <= app_order_id]
        for i in range(len(order_data_a_new)):
            if order_data_a_new[i]['app_order_id'] == app_order_id:
                order_data_a_new[i]['status'] = 20
        return order_data_a_new
#order_data_a = preprocess_order_data_a(main_order_id, order_data_a)


In [82]:
def preprocess_installments_data_a(installments_data_a, contract_data_a, apply_time, app_order_id):
    """
    预处理installments_data_a数据，仅回溯时使用。
    :param installments_data_a:原始数据
    :param contract_data_a:合同表
    :param apply_time:申请时间
    :param app_order_id:申请订单号
    :return:
    """

    app_order_id = int(app_order_id)

    if len(installments_data_a) == 0:
        return installments_data_a

    else:
        df_installments_data_a = pd.DataFrame(installments_data_a)
        df_contract_data_a = pd.DataFrame(contract_data_a)

        df_installments_data_a = df_installments_data_a[df_installments_data_a['app_order_id'] <= app_order_id]
        # df_installments_data_a = df_installments_data_a[df_installments_data_a['status'] != -1]

        df_installments_data_a.sort_values(by='app_order_id', inplace=True)
        df_installments_data_a.index = range(df_installments_data_a.shape[0])
        df_installments_data_a = pd.merge(df_installments_data_a,
                                          df_contract_data_a[['app_order_id', 'settlement_time']], on='app_order_id',
                                          how='left', suffixes=('_drop', ''))

        filtered_data = df_installments_data_a[df_installments_data_a['settlement_time'] > apply_time]
        # 找到重复的 app_order_id，并获取对应的 extension_count 最大值
        duplicate_mask = filtered_data['app_order_id'].duplicated(keep=False)
        max_extension_count = filtered_data.groupby('app_order_id')['extension_count'].transform('max')
        # 更新 status 为 1 的条件
        condition = (duplicate_mask) & (filtered_data['extension_count'] == max_extension_count.values)
        condition1 = ~duplicate_mask

        # 将满足条件的列更新数值
        filtered_data.loc[condition, 'status'] = 1
        filtered_data.loc[condition1, 'status'] = 1
        filtered_data.loc[condition, 'settlement_type'] = 0
        filtered_data.loc[condition1, 'settlement_type'] = 0
        filtered_data.loc[condition, 'extension_fee'] = 0.0
        filtered_data.loc[condition1, 'extension_fee'] = 0.0
        df_installments_data_a.loc[df_installments_data_a.index.isin(filtered_data.index), 'status'] = filtered_data[
            'status']
        df_installments_data_a.loc[df_installments_data_a.index.isin(filtered_data.index), 'settlement_type'] = \
            filtered_data['settlement_type']
        df_installments_data_a.loc[df_installments_data_a.index.isin(filtered_data.index), 'extension_fee'] = \
            filtered_data['extension_fee']

        df_installments_data_a.loc[df_installments_data_a['settlement_time'] > apply_time, 'repaid_principal'] = 0.0
        df_installments_data_a.loc[df_installments_data_a['settlement_time'] > apply_time, 'repaid_interest'] = 0.0
        # df_installments_data_a.loc[df_installments_data_a['settlement_time'] > apply_time, 'repaid_cut_interest'] = 0.0
        df_installments_data_a.loc[df_installments_data_a['settlement_time'] > apply_time, 'repaid_service_fee'] = 0.0
        df_installments_data_a.loc[
            df_installments_data_a['settlement_time'] > apply_time, 'repaid_management_fee'] = 0.0
        df_installments_data_a.loc[
            df_installments_data_a['settlement_time'] > apply_time, 'repaid_overdue_interest'] = 0.0
        df_installments_data_a.loc[df_installments_data_a['settlement_time'] > apply_time, 'repaid_penalty'] = 0.0

        df_installments_data_a['overdue_interest'] = 0.0
        df_installments_data_a['overdue_days'] = 0.0
        # df_installments_data_a.fillna(-999, inplace=True)
        # df_installments_data_a['update_time'] = df_installments_data_a['settlement_time']

        installments_data_a_new = df_installments_data_a.to_dict(orient='records')
        # installments_data_a_new = [item for item in installments_data_a if item['app_order_id'] <= app_order_id]

        return installments_data_a_new


In [111]:
#将查询的数据进行拼接组装（assemble_req_data）
req_data_list = []
logger.info(f"开始组装数据,共计{len(data_info)}条数据")
for i, v in tqdm(data_info.iterrows(), desc='assemble data', disable=False):
    main_user_id = str(v['app_user_id'])
    main_order_id = str(v['order_id'])
    main_apply_time = v['apply_time']  # 实际为req表的请求时间一般大于订单表的时间，所以会包含本单
    main_nid, main_phone = str(v['nid']), str(v['phone'])
    try:
        main_tx_id = str(v['tx_id'])
        req_obj = {}
        apply_time = str(v['real_apply_time'])
        # 兼容source异常的情况
        source = v.get('source', -1)
        if source is None:
            source = -999
        source = int(source)
        #构建基础信息
        base_info = {'country_code': v['country_code'],
                     'merchant_id': v['merchant_id'],
                     'country_abbr': v['country_abbr'],
                     'tx_id': str(v['tx_id']),
                     'order_id': str(v['order_id']),
                     'phone': main_phone,
                     'nid': main_nid,
                     'source': source,
                     'acq_channel': v['acq_channel'],
                     'device_type': str(v['device_type']),
                     'app_user_id': str(v['app_user_id']),
                     'apply_time': apply_time,
                     'track_id': str(v['track_id'])}
        user_id = v['app_user_id']  #过滤数据？

        req_obj['base_info'] = base_info#？？
        #处理时间格式
        order_data['apply_time'] = order_data.apply_time.apply(lambda x: None if pd.isna(x) else  str(x) )
        order_data['create_time'] = order_data.create_time.apply(lambda x: None if pd.isna(x) else  str(x))
        order_data['update_time'] = order_data.update_time.apply(lambda x: None if pd.isna(x) else  str(x))

        installment_data['create_time'] = installment_data.create_time.apply(lambda x: None if pd.isna(x) else  str(x))
        installment_data['update_time'] = installment_data.update_time.apply(lambda x: None if pd.isna(x) else  str(x))
        installment_data['repayment_date'] = installment_data.repayment_date.apply(lambda x: None if pd.isna(x) else  str(x))
        installment_data['settlement_time'] = installment_data.settlement_time.apply(lambda x: None if pd.isna(x) else str(x))

        contract_data['settlement_time'] = contract_data.settlement_time.apply(lambda x: None if pd.isna(x) else  str(x))
        contract_data['create_time'] = contract_data.create_time.apply(lambda x: None if pd.isna(x) else  str(x))
        contract_data['update_time'] = contract_data.update_time.apply(lambda x: None if pd.isna(x) else  str(x))
        contract_data['activation_time'] = contract_data.activation_time.apply(lambda x: None if pd.isna(x) else  str(x))

        device_list_data['update_time'] = device_list_data.update_time.apply(lambda x: None if pd.isna(x) else  str(x) )
        device_list_data['day_inter'] = (
                pd.to_datetime(apply_time) - pd.to_datetime(device_list_data['update_time'])).dt.days

        #过滤相关数据
        installment_data_a = installment_data[
            (installment_data.user_id == user_id) & (installment_data.create_time < apply_time)]
        order_data_a = order_data[
            (order_data.user_id == user_id) & (order_data.apply_time <= apply_time)]  # 包含等号，需要同一批次审核单
        contract_data_a = contract_data[
            (contract_data.user_id == user_id) & (contract_data.create_time <= apply_time)]
        device_list_data_a = device_list_data[
            (device_list_data.user_id == user_id) & (device_list_data.day_inter <= 180) & (
                    apply_time >= device_list_data.update_time)]

        user_info = v['user_info']
        device_info = v['device_info']
        installment_data_a = json.loads(installment_data_a.to_json(orient='records'))
        order_data_a = json.loads(order_data_a.to_json(orient='records'))
        contract_data_a = json.loads(contract_data_a.to_json(orient='records'))
        device_list_data_a = json.loads(device_list_data_a.to_json(orient='records'))

        if len(installment_data_a) == 0:
            installment_data_a = []
        if len(order_data_a) == 0:
            order_data_a = []
        if len(contract_data_a) == 0:
            contract_data_a = []
        if len(device_list_data_a) == 0:
            device_list_data_a = []

        applist_data = [] if v['app_list_url_data'] is None else json.loads(v['app_list_url_data'])
        sms_data = [] if v['sms_records_url_data'] is None else json.loads(v['sms_records_url_data'])
        calllog_data = [] if v['call_records_url_data'] is None else json.loads(v['call_records_url_data'])
        calendar_data = [] if v['calendar_records_url_data'] is None else json.loads(v['calendar_records_url_data'])
        try:
            contact_list = user_info['userInfo']['userContactInfo']
        except:
            contact_list = []

        str_apply_time = str(main_apply_time)
        order_data_a = preprocess_order_data_a(main_order_id, order_data_a)
        installment_data_a = preprocess_installments_data_a(installment_data_a, contract_data_a, str_apply_time,
                                                            main_order_id)
        
        #构建请求的对象
        #req_obj['data_sources'] = {'user_info_1': user_info,
                                  # 'contact_list': contact_list,
                                   #'device_info': device_info,
                                   #'order_data_a': order_data_a,
                                   #'installments_data_a': installment_data_a,
                                   #'contract_data_a': contract_data_a,
                                   #'applist_data': applist_data,
                                   #'sms_data': sms_data,
                                   #'calllog_data': calllog_data,
                                   #'calendar_data': calendar_data,
                                   #'device_list': device_list_data_a}
        
        #【loan_th_base_v3_2】:'user_info_1','order_data_a','installments_data_a','contract_data_a'
        #req_obj['data_sources'] = {'user_info_1': user_info,'order_data_a': order_data_a,'installments_data_a': installment_data_a,'contract_data_a': contract_data_a,}
        #sms_th_base_v2:'sms_data'
        req_obj['data_sources'] = {'sms_data': sms_data}
        #添加fdc数据
        if fdc_data is not None:
            order_fdc_data = fdc_data[fdc_data['order_id'] == main_order_id]
            if not order_fdc_data.empty:
                req_obj['data_sources'].update({'fdc_id_inquiryV5': order_fdc_data['data'].iloc[0]})
        #合并进req_data_list：
        req_data_list.append(
            [main_nid, main_phone, main_user_id, main_order_id, main_tx_id, main_apply_time,
             json.dumps(req_obj, ensure_ascii=False)])
    except:
        logger.info(f"订单{main_order_id},封装时发生异常:\n{traceback.format_exc()}")
        continue


2025-05-07 12:07:30,126 - toollib.data_fetcher - INFO - 开始组装数据,共计10条数据
assemble data: 10it [00:00, 13.48it/s]


In [112]:
#生成组装后的dataframe        
req_df = pd.DataFrame(req_data_list,columns=['nid', 'phone', 'app_user_id', 'app_order_id', 'tx_id', 'apply_time', 'request_params'])

In [113]:
df_source = req_df

In [114]:
df_source 

,nid,phone,app_user_id,app_order_id,tx_id,apply_time,request_params
0,1310800193097,0930188843,1318174204879134720,1323669295601246208,1323677179810635776,2024-12-31 22:09:12,"{""base_info"": {""country_code"": ""66"", ""merchant..."
1,3320501227930,0616546325,1310220729914449920,1323682197271961600,1323690013953056768,2024-12-31 23:00:28,"{""base_info"": {""country_code"": ""66"", ""merchant..."
2,3209600232507,0854381432,1317640458664894464,1323701931170291712,1323701968784809984,2025-01-01 00:18:54,"{""base_info"": {""country_code"": ""66"", ""merchant..."
3,3740300555570,0945540414,1261698652018532352,1323702219562246144,1323702345286508544,2025-01-01 00:20:02,"{""base_info"": {""country_code"": ""66"", ""merchant..."
4,3100800873461,0854855085,1258062236822757376,1323703298987352064,1323703393522769921,2025-01-01 00:24:22,"{""base_info"": {""country_code"": ""66"", ""merchant..."
5,3100800873461,0854855085,1258062236822757376,1323706215282728960,1323706248283512832,2025-01-01 00:35:58,"{""base_info"": {""country_code"": ""66"", ""merchant..."
6,3740200077714,0818099792,1302890098922577920,1323713280021585920,1323713455247024128,2025-01-01 01:04:05,"{""base_info"": {""country_code"": ""66"", ""merchant..."
7,3740200077714,0818099792,1302890098922577920,1323713280029974528,1323713437102465024,2025-01-01 01:04:05,"{""base_info"": {""country_code"": ""66"", ""merchant..."
8,3101202761112,0992292953,1320966367341928448,1323722163423240192,1323722226748841984,2025-01-01 01:39:24,"{""base_info"": {""country_code"": ""66"", ""merchant..."
9,3101202761112,0992292953,1320966367341928448,1323722163427434496,1323722244989870080,2025-01-01 01:39:24,"{""base_info"": {""country_code"": ""66"", ""merchant..."


In [87]:
req_df

,nid,phone,app_user_id,app_order_id,tx_id,apply_time,request_params
0,1310800193097,0930188843,1318174204879134720,1323669295601246208,1323677179810635776,2024-12-31 22:09:12,"{""base_info"": {""country_code"": ""66"", ""merchant..."
1,3320501227930,0616546325,1310220729914449920,1323682197271961600,1323690013953056768,2024-12-31 23:00:28,"{""base_info"": {""country_code"": ""66"", ""merchant..."
2,3209600232507,0854381432,1317640458664894464,1323701931170291712,1323701968784809984,2025-01-01 00:18:54,"{""base_info"": {""country_code"": ""66"", ""merchant..."
3,3740300555570,0945540414,1261698652018532352,1323702219562246144,1323702345286508544,2025-01-01 00:20:02,"{""base_info"": {""country_code"": ""66"", ""merchant..."
4,3100800873461,0854855085,1258062236822757376,1323703298987352064,1323703393522769921,2025-01-01 00:24:22,"{""base_info"": {""country_code"": ""66"", ""merchant..."
5,3100800873461,0854855085,1258062236822757376,1323706215282728960,1323706248283512832,2025-01-01 00:35:58,"{""base_info"": {""country_code"": ""66"", ""merchant..."
6,3740200077714,0818099792,1302890098922577920,1323713280021585920,1323713455247024128,2025-01-01 01:04:05,"{""base_info"": {""country_code"": ""66"", ""merchant..."
7,3740200077714,0818099792,1302890098922577920,1323713280029974528,1323713437102465024,2025-01-01 01:04:05,"{""base_info"": {""country_code"": ""66"", ""merchant..."
8,3101202761112,0992292953,1320966367341928448,1323722163423240192,1323722226748841984,2025-01-01 01:39:24,"{""base_info"": {""country_code"": ""66"", ""merchant..."
9,3101202761112,0992292953,1320966367341928448,1323722163427434496,1323722244989870080,2025-01-01 01:39:24,"{""base_info"": {""country_code"": ""66"", ""merchant..."


In [14]:
#df_source = get_sample_req(app_order_ids,'ath',user, passwd)

2025-05-07 10:02:54,025 - toollib.asystem_env.sample_util - INFO - 共计输入2条数据，清理重复数据后共2条数据
2025-05-07 10:02:54,460 - toollib.asystem_env.sample_util - INFO - 成功查询到进件相关信息,共计(2, 23),对应的用户数量2
2025-05-07 10:02:54,462 - toollib.asystem_env.sample_util - INFO - 剔除异常订单后，还剩余2订单，对应2个用户
2025-05-07 10:02:54,871 - toollib.asystem_env.sample_util - INFO - 基于user_id关联的放款单,共计(112, 34)
2025-05-07 10:02:55,411 - toollib.asystem_env.sample_util - INFO - 基于user_id关联的进件订单,共计(129, 34)
2025-05-07 10:02:55,504 - toollib.asystem_env.sample_util - INFO - 基于user_id关联的合同单,共计(116, 40)
2025-05-07 10:02:55,592 - toollib.asystem_env.sample_util - INFO - 基于user_id关联设备数据,共计(101, 4)
calendar: 100%|██████████| 2/2 [00:00<00:00, 5886.74it/s]
2025-05-07 10:02:56,890 - toollib.asystem_env.sample_util - INFO - 开始组装数据,共计2条数据
assemble data: 2it [00:00,  9.96it/s]


In [16]:
#df_source

,nid,phone,app_user_id,app_order_id,tx_id,apply_time,request_params
0,1310800193097,0930188843,1318174204879134720,1323669295601246208,1323677179810635776,2024-12-31 22:09:12,"{""base_info"": {""country_code"": ""66"", ""merchant..."
1,3320501227930,0616546325,1310220729914449920,1323682197271961600,1323690013953056768,2024-12-31 23:00:28,"{""base_info"": {""country_code"": ""66"", ""merchant..."
2,3209600232507,0854381432,1317640458664894464,1323701931170291712,1323701968784809984,2025-01-01 00:18:54,"{""base_info"": {""country_code"": ""66"", ""merchant..."
3,3740300555570,0945540414,1261698652018532352,1323702219562246144,1323702345286508544,2025-01-01 00:20:02,"{""base_info"": {""country_code"": ""66"", ""merchant..."
4,3100800873461,0854855085,1258062236822757376,1323703298987352064,1323703393522769921,2025-01-01 00:24:22,"{""base_info"": {""country_code"": ""66"", ""merchant..."
5,3100800873461,0854855085,1258062236822757376,1323706215282728960,1323706248283512832,2025-01-01 00:35:58,"{""base_info"": {""country_code"": ""66"", ""merchant..."
6,3740200077714,0818099792,1302890098922577920,1323713280021585920,1323713455247024128,2025-01-01 01:04:05,"{""base_info"": {""country_code"": ""66"", ""merchant..."
7,3740200077714,0818099792,1302890098922577920,1323713280029974528,1323713437102465024,2025-01-01 01:04:05,"{""base_info"": {""country_code"": ""66"", ""merchant..."
8,3101202761112,0992292953,1320966367341928448,1323722163423240192,1323722226748841984,2025-01-01 01:39:24,"{""base_info"": {""country_code"": ""66"", ""merchant..."
9,3101202761112,0992292953,1320966367341928448,1323722163427434496,1323722244989870080,2025-01-01 01:39:24,"{""base_info"": {""country_code"": ""66"", ""merchant..."


In [15]:
#js

{'base_info': {'country_code': '66',
  'merchant_id': 'ATH',
  'country_abbr': 'th',
  'tx_id': '1323677179810635776',
  'order_id': '1323669295601246208',
  'phone': '0930188843',
  'nid': '1310800193097',
  'source': 1,
  'acq_channel': 'VIJG',
  'device_type': '0',
  'app_user_id': '1318174204879134720',
  'apply_time': '2024-12-31 22:40:32',
  'track_id': '2300467'},
 'data_sources': {'user_info_1': {'userInfo': {'ocrRecord': {'code': 'SUCCESS',
     'gender': '1',
     'birthday': '14-05-1996',
     'cardSide': 'FRONT',
     'fullName': 'Mr Yodsaton',
     'idNumber': '1310800193097',
     'isNormal': 1,
     'appUserId': '1318174204879134720',
     'photoPath': 'https://bienslcx.ueuti.com/file/download/404a42a7-bf27-4809-a8e4-7364a04848d4.jpg',
     'acqChannel': 'VIJG',
     'createTime': '2024-12-16 18:14:46',
     'updateTime': '2024-12-16 18:14:46',
     'partnerCode': 'accu',
     'phoneNumber': '0930188843',
     'transactionId': 'TID85fc8436f46f412e832e1c3f88d7dec6',
     

In [126]:
js = json.loads(df_source['request_params'][0])

In [127]:
js['data_sources'].keys()

dict_keys(['sms_data'])

In [ ]:
#loan_th_base_v3_2:'user_info_1','order_data_a','installments_data_a','contract_data_a'

In [117]:
js['data_sources'].keys()   #查询js中字典{data_sources}的主键

dict_keys(['sms_data'])

In [128]:
#填写对应的特征模块接口，以及对应模块名称(注意需要填写模块名称)
#flask_url = 'http://127.0.0.1:5123/api/loan_th_base_v3_2'
flask_url = 'http://127.0.0.1:5121/api/sms_th_base_v2'


In [129]:
#填写对应sql语句,从数据库里拼接原始数据源
req_data = js

In [130]:
#向接口发送请求，传输对应格式的数据。
response = requests.post(flask_url, json=req_data)

In [131]:
response.json().keys()

dict_keys(['code', 'data'])

In [132]:
response.json()

{'code': 200,
 'data': {'sms_th_base_v2_all_phone_apply_first_days_diff_15d': 14.0,
  'sms_th_base_v2_all_phone_apply_first_days_diff_15d_30d_diff': -15.0,
  'sms_th_base_v2_all_phone_apply_first_days_diff_15d_30d_ratio': 0.4828,
  'sms_th_base_v2_all_phone_apply_first_days_diff_180d': 170.0,
  'sms_th_base_v2_all_phone_apply_first_days_diff_180d_infd_diff': 0.0,
  'sms_th_base_v2_all_phone_apply_first_days_diff_180d_infd_ratio': 1.0,
  'sms_th_base_v2_all_phone_apply_first_days_diff_1d': 1.0,
  'sms_th_base_v2_all_phone_apply_first_days_diff_1d_3d_diff': -2.0,
  'sms_th_base_v2_all_phone_apply_first_days_diff_1d_3d_ratio': 0.3333,
  'sms_th_base_v2_all_phone_apply_first_days_diff_30d': 29.0,
  'sms_th_base_v2_all_phone_apply_first_days_diff_30d_60d_diff': -15.0,
  'sms_th_base_v2_all_phone_apply_first_days_diff_30d_60d_ratio': 0.6591,
  'sms_th_base_v2_all_phone_apply_first_days_diff_3d': 3.0,
  'sms_th_base_v2_all_phone_apply_first_days_diff_3d_7d_diff': -4.0,
  'sms_th_base_v2_all_p